In [ ]:
# ---- Configuration: edit config/config.yaml (see README) ----
# Correlation plots: protein levels vs clinical score, incl. high NfH / Abeta42 subsets
# Input files are read from, and outputs written to, <correlation_workdir> (see README for the file list).
import os, sys
from pathlib import Path
_root = Path(os.environ.get("PROJECT_ROOT") or next(
    p for p in [Path.cwd(), *Path.cwd().parents] if (p / "utils" / "config.py").exists()))
os.environ["PROJECT_ROOT"] = str(_root)
sys.path.insert(0, str(_root))
from utils.config import cfg, cfg_path
os.chdir(cfg_path("correlation_workdir", create=True))


In [ ]:
import pandas as pd
import numpy as np

In [ ]:
df1 = pd.read_excel("correlation.xlsx", sheet_name=0, index_col=0)  # 0 refers to the first sheet
df1.fillna(0, inplace=True)
df1

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import pearsonr

# Assuming df1 is your DataFrame containing the data

# Plot the scatter plot with regression line
plt.figure(figsize=(8, 6))
sns.regplot(x=df1['pNfH 3-BL'], y=df1['UCHL1 3-BL'], scatter_kws={'alpha': 0.5}, line_kws={'color': 'black'}, color='blue')

# Calculate Pearson correlation coefficient and p-value
pearson_corr, p_value = pearsonr(df1['pNfH 3-BL'], df1['UCHL1 3-BL'])

# Calculate R-squared value
r_squared = pearson_corr**2

# Add Pearson correlation coefficient, p-value, and R-squared value as text
plt.text(df1['pNfH 3-BL'].min(), df1['UCHL1 3-BL'].max(), 
         'Pearson Correlation: {:.2f}\nP-value: {:.8f}\nR²: {:.2f}'.format(pearson_corr, p_value, r_squared),
         verticalalignment='bottom', horizontalalignment='left')

# Add labels to each point based on the index
for index, _ in df1.iterrows():
    plt.text(df1.loc[index, 'pNfH 3-BL'], df1.loc[index, 'UCHL1 3-BL'], str(index), fontsize=8)

plt.title('Scatter Plot')
plt.xlabel('pNfH 3-BL')
plt.ylabel('UCHL1 3-BL')

plt.savefig('df1.png')

# Show plot
plt.show()


In [ ]:
import numpy as np
from scipy.stats import pearsonr
import seaborn as sns
import matplotlib.pyplot as plt

# Bootstrapping function for Pearson correlation coefficient
def bootstrap_pearsonr(data_x, data_y, n_bootstraps=1000, ci=95):
    bootstrapped_corrs = []
    for _ in range(n_bootstraps):
        sample_indices = np.random.choice(len(data_x), len(data_x), replace=True)
        sample_x = data_x[sample_indices]
        sample_y = data_y[sample_indices]
        boot_corr, _ = pearsonr(sample_x, sample_y)
        bootstrapped_corrs.append(boot_corr)
    lower_bound = np.percentile(bootstrapped_corrs, (100 - ci) / 2)
    upper_bound = np.percentile(bootstrapped_corrs, 100 - (100 - ci) / 2)
    return lower_bound, upper_bound

# Assuming df1 is your DataFrame containing the data
data_x = df1['pNfH 3-BL'].values
data_y = df1['UCHL1 3-BL'].values

# Calculate Pearson correlation coefficient and p-value
pearson_corr, p_value = pearsonr(data_x, data_y)

# Calculate R-squared value
r_squared = pearson_corr**2

# Calculate confidence intervals
ci_lower, ci_upper = bootstrap_pearsonr(data_x, data_y)

# Plot the scatter plot with regression line
plt.figure(figsize=(8, 6))
sns.regplot(x=data_x, y=data_y, scatter_kws={'alpha': 0.5}, line_kws={'color': 'black'}, color='blue')

# Add Pearson correlation coefficient, p-value, R-squared value, and CI as text
plt.text(min(data_x), max(data_y), 
         f'Pearson Correlation: {pearson_corr:.2f}\n'
         f'P-value: {p_value:.8f}\n'
         f'R²: {r_squared:.2f}\n'
         f'95% CI: [{ci_lower:.2f}, {ci_upper:.2f}]',
         verticalalignment='bottom', horizontalalignment='left')

# Add labels to each point based on the index
for index, _ in df1.iterrows():
    plt.text(df1.loc[index, 'pNfH 3-BL'], df1.loc[index, 'UCHL1 3-BL'], str(index), fontsize=8)

plt.title('Scatter Plot')
plt.xlabel('pNfH 3-BL')
plt.ylabel('UCHL1 3-BL')

plt.savefig('df1_with_CI.png')

# Show plot
plt.show()


In [ ]:
import numpy as np
from scipy.stats import pearsonr, f_oneway
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression

# Bootstrapping function for Pearson correlation coefficient
def bootstrap_pearsonr(data_x, data_y, n_bootstraps=1000, ci=95):
    bootstrapped_corrs = []
    for _ in range(n_bootstraps):
        sample_indices = np.random.choice(len(data_x), len(data_x), replace=True)
        sample_x = data_x[sample_indices]
        sample_y = data_y[sample_indices]
        boot_corr, _ = pearsonr(sample_x, sample_y)
        bootstrapped_corrs.append(boot_corr)
    lower_bound = np.percentile(bootstrapped_corrs, (100 - ci) / 2)
    upper_bound = np.percentile(bootstrapped_corrs, 100 - (100 - ci) / 2)
    return lower_bound, upper_bound

# Assuming df1 is your DataFrame containing the data
data_x = df1['pNfH 3-BL'].values.reshape(-1, 1)
data_y = df1['UCHL1 3-BL'].values

# Linear regression model for adjusted R²
model = LinearRegression()
model.fit(data_x, data_y)
r_squared = model.score(data_x, data_y)
n = len(data_x)  # Number of samples
p = 1  # Number of predictors
adjusted_r_squared = 1 - ((1 - r_squared) * (n - 1) / (n - p - 1))

# F-test calculation
f_statistic, f_p_value = f_oneway(data_x.flatten(), data_y)

# Calculate Pearson correlation coefficient and p-value
pearson_corr, p_value = pearsonr(data_x.flatten(), data_y)

# Calculate confidence intervals
ci_lower, ci_upper = bootstrap_pearsonr(data_x.flatten(), data_y)

# Plot the scatter plot with regression line
plt.figure(figsize=(8, 8))
sns.regplot(x=data_x.flatten(), y=data_y, scatter_kws={'alpha': 0.5}, line_kws={'color': 'black'}, color='blue')

# Add labels to each point based on the index
for index, _ in df1.iterrows():
    plt.text(df1.loc[index, 'pNfH 3-BL'], df1.loc[index, 'UCHL1 3-BL'], str(index), fontsize=8)

# Add Pearson correlation coefficient, p-value, R², adjusted R², F-statistic, and CI as a table
table_data = [
    ['Pearson Correlation', f'{pearson_corr:.2f}'],
    ['P-value', f'{p_value:.8f}'],
    ['R²', f'{r_squared:.2f}'],
    ['Adjusted R²', f'{adjusted_r_squared:.2f}'],
    ['F-statistic', f'{f_statistic:.2f}'],
    ['F-test P-value', f'{f_p_value:.8f}'],
    ['95% CI', f'[{ci_lower:.2f}, {ci_upper:.2f}]']
]

# Add the table below the plot
plt.table(cellText=table_data,
          colLabels=['Statistic', 'Value'],
          cellLoc='center', 
          loc='bottom', 
          bbox=[0.0, -0.4, 1, 0.3])  # Adjust bbox to position the table

plt.title('Scatter Plot with Statistics')
plt.xlabel('pNfH 3-BL')
plt.ylabel('UCHL1 3-BL')

plt.tight_layout()
plt.savefig('scatter_with_stats_and_labels.png')

# Show plot
plt.show()


In [ ]:
df2 = pd.read_excel("correlation.xlsx", sheet_name=1, index_col=0)  # 0 refers to the first sheet
df2.fillna(0, inplace=True)
df2

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import pearsonr

# Assuming df1 is your DataFrame containing the data

# Plot the scatter plot with regression line and rho value
plt.figure(figsize=(8, 6))
sns.regplot(x=df2['pNfH 6-3'], y=df2['UCHL1 6-3'], scatter_kws={'alpha':0.5}, line_kws={'color': 'black'}, color='blue')

# Calculate Pearson correlation coefficient and p-value
pearson_corr, p_value = pearsonr(df2['pNfH 6-3'], df2['UCHL1 6-3'])

# Calculate R-squared value
r_squared = pearson_corr**2

# Add Pearson correlation coefficient, p-value, and R-squared value as text
plt.text(df2['pNfH 6-3'].min(), df2['UCHL1 6-3'].max(), 
         'Pearson Correlation: {:.2f}\nP-value: {:.8f}\nR²: {:.2f}'.format(pearson_corr, p_value, r_squared),
         verticalalignment='bottom', horizontalalignment='left')

# Add labels to each point based on the index
for index, _ in df2.iterrows():
    plt.text(df2.loc[index, 'pNfH 6-3'], df2.loc[index, 'UCHL1 6-3'], str(index), fontsize=8)

plt.title('Scatter Plot')
plt.xlabel('pNfH 6-3')
plt.ylabel('UCHL1 6-3')

plt.savefig('df2.png')

# Show plot
plt.show()


In [ ]:
df3 = pd.read_excel("correlation.xlsx", sheet_name=2, index_col=0)  # 0 refers to the first sheet
df3.fillna(0, inplace=True)
df3

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import pearsonr

# Assuming df1 is your DataFrame containing the data

# Plot the scatter plot with regression line and rho value
plt.figure(figsize=(8, 6))
sns.regplot(x=df3['pNfH 6-BL'], y=df3['UCHL1 6-BL'], scatter_kws={'alpha':0.5}, line_kws={'color': 'black'}, color='blue')

# Calculate Pearson correlation coefficient and p-value
pearson_corr, p_value = pearsonr(df3['pNfH 6-BL'], df3['UCHL1 6-BL'])

# Calculate R-squared value
r_squared = pearson_corr**2

# Add Pearson correlation coefficient, p-value, and R-squared value as text
plt.text(df3['pNfH 6-BL'].min(), df3['UCHL1 6-BL'].max(), 
         'Pearson Correlation: {:.2f}\nP-value: {:.8f}\nR²: {:.2f}'.format(pearson_corr, p_value, r_squared),
         verticalalignment='top', horizontalalignment='left')

# Add labels to each point based on the index
for index, _ in df3.iterrows():
    plt.text(df3.loc[index, 'pNfH 6-BL'], df3.loc[index, 'UCHL1 6-BL'], str(index), fontsize=8)

plt.title('Scatter Plot')
plt.xlabel('pNfH 6-BL')
plt.ylabel('UCHL1 6-BL')

plt.savefig('df3.png')

# Show plot
plt.show()

In [ ]:
df4 = pd.read_excel("correlation.xlsx", sheet_name=3, index_col=0)  # 0 refers to the first sheet
#df4.fillna(0, inplace=True)
df4

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import pearsonr

# Assuming df1 is your DataFrame containing the data

# Plot the scatter plot with regression line and rho value
plt.figure(figsize=(8, 6))
sns.regplot(x=df4['Score-R 6-3'], y=df4['UCHL1 6-3'], scatter_kws={'alpha':0.5}, line_kws={'color': 'black'}, color='blue')

# Calculate Pearson correlation coefficient and p-value
pearson_corr, p_value = pearsonr(df4['Score-R 6-3'], df4['UCHL1 6-3'])

# Calculate R-squared value
r_squared = pearson_corr**2

# Add Pearson correlation coefficient, p-value, and R-squared value as text
plt.text(df4['Score-R 6-3'].min(), df4['UCHL1 6-3'].max(), 
         'Pearson Correlation: {:.2f}\nP-value: {:.8f}\nR²: {:.2f}'.format(pearson_corr, p_value, r_squared),
         verticalalignment='top', horizontalalignment='left')

# Add labels to each point based on the index
for index, _ in df4.iterrows():
    plt.text(df4.loc[index, 'Score-R 6-3'], df4.loc[index, 'UCHL1 6-3'], str(index), fontsize=8)

plt.title('Scatter Plot')
plt.xlabel('Score-R 6-3')
plt.ylabel('UCHL1 6-3')

plt.savefig('df4.png')

# Show plot
plt.show()

In [ ]:
df5 = pd.read_excel("correlation.xlsx", sheet_name=4, index_col=0)  # 0 refers to the first sheet
df5.fillna(0, inplace=True)
df5

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import pearsonr

# Assuming df1 is your DataFrame containing the data

# Plot the scatter plot with regression line and rho value
plt.figure(figsize=(8, 6))
sns.regplot(x=df5['pNfH 6-BL'], y=df5['Aß 1-40 6-BL'], scatter_kws={'alpha':0.5}, line_kws={'color': 'black'}, color='blue')

# Calculate Pearson correlation coefficient and p-value
pearson_corr, p_value = pearsonr(df5['pNfH 6-BL'], df5['Aß 1-40 6-BL'])

# Calculate R-squared value
r_squared = pearson_corr**2

# Add Pearson correlation coefficient, p-value, and R-squared value as text
plt.text(df5['pNfH 6-BL'].min(), df5['Aß 1-40 6-BL'].max(), 
         'Pearson Correlation: {:.2f}\nP-value: {:.8f}\nR²: {:.2f}'.format(pearson_corr, p_value, r_squared),
         verticalalignment='top', horizontalalignment='left')

# Add labels to each point based on the index
for index, _ in df5.iterrows():
    plt.text(df5.loc[index, 'pNfH 6-BL'], df5.loc[index, 'Aß 1-40 6-BL'], str(index), fontsize=8)

plt.title('Scatter Plot')
plt.xlabel('pNfH 6-BL')
plt.ylabel('Aß 1-40 6-BL')

plt.savefig('df5.png')

# Show plot
plt.show()

In [ ]:
df6 = pd.read_excel("correlation.xlsx", sheet_name=5, index_col=0)  # 0 refers to the first sheet
df6.fillna(0, inplace=True)
df6

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import pearsonr

# Assuming df1 is your DataFrame containing the data

# Plot the scatter plot with regression line and rho value
plt.figure(figsize=(8, 6))
sns.regplot(x=df6['pNfH 6-BL'], y=df6['Aß 1-42 6-BL'], scatter_kws={'alpha':0.5}, line_kws={'color': 'black'}, color='blue')

# Calculate Pearson correlation coefficient and p-value
pearson_corr, p_value = pearsonr(df6['pNfH 6-BL'], df6['Aß 1-42 6-BL'])

# Calculate R-squared value
r_squared = pearson_corr**2

# Add Pearson correlation coefficient, p-value, and R-squared value as text
plt.text(df6['pNfH 6-BL'].min(), df6['Aß 1-42 6-BL'].max(), 
         'Pearson Correlation: {:.2f}\nP-value: {:.8f}\nR²: {:.2f}'.format(pearson_corr, p_value, r_squared),
         verticalalignment='top', horizontalalignment='left')

# Add labels to each point based on the index
for index, _ in df6.iterrows():
    plt.text(df6.loc[index, 'pNfH 6-BL'], df6.loc[index, 'Aß 1-42 6-BL'], str(index), fontsize=8)

plt.title('Scatter Plot')
plt.xlabel('pNfH 6-BL')
plt.ylabel('Aß 1-42 6-BL')

plt.savefig('df6.png')

# Show plot
plt.show()

In [ ]:
df7 = pd.read_excel("correlation.xlsx", sheet_name=6, index_col=0)  # 0 refers to the first sheet
df7.fillna(0, inplace=True)
df7

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import pearsonr

# Assuming df1 is your DataFrame containing the data

# Plot the scatter plot with regression line and rho value
plt.figure(figsize=(8, 6))
sns.regplot(x=df7['Score-R 3-BL'], y=df7['Aß 1-40 3-BL'], scatter_kws={'alpha':0.5}, line_kws={'color': 'black'}, color='blue')

# Calculate Pearson correlation coefficient and p-value
pearson_corr, p_value = pearsonr(df7['Score-R 3-BL'], df7['Aß 1-40 3-BL'])

# Calculate R-squared value
r_squared = pearson_corr**2

# Add Pearson correlation coefficient, p-value, and R-squared value as text
plt.text(df7['Score-R 3-BL'].min(), df7['Aß 1-40 3-BL'].max(), 
         'Pearson Correlation: {:.2f}\nP-value: {:.8f}\nR²: {:.2f}'.format(pearson_corr, p_value, r_squared),
         verticalalignment='top', horizontalalignment='left')

# Add labels to each point based on the index
for index, _ in df7.iterrows():
    plt.text(df7.loc[index, 'Score-R 3-BL'], df7.loc[index, 'Aß 1-40 3-BL'], str(index), fontsize=8)

plt.title('Scatter Plot')
plt.xlabel('Score-R 3-BL')
plt.ylabel('Aß 1-40 3-BL')

plt.savefig('df7.png')

# Show plot
plt.show()

In [ ]:
df8 = pd.read_excel("correlation.xlsx", sheet_name=7, index_col=0)  # 0 refers to the first sheet
df8.fillna(0, inplace=True)
df8

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import pearsonr

# Assuming df1 is your DataFrame containing the data

# Plot the scatter plot with regression line and rho value
plt.figure(figsize=(8, 6))
sns.regplot(x=df8['Score-R 3-BL'], y=df8['Aß 1-42 3-BL'], scatter_kws={'alpha':0.5}, line_kws={'color': 'black'}, color='blue')

# Calculate Pearson correlation coefficient and p-value
pearson_corr, p_value = pearsonr(df8['Score-R 3-BL'], df8['Aß 1-42 3-BL'])

# Calculate R-squared value
r_squared = pearson_corr**2

# Add Pearson correlation coefficient, p-value, and R-squared value as text
plt.text(df8['Score-R 3-BL'].min(), df8['Aß 1-42 3-BL'].max(), 
         'Pearson Correlation: {:.2f}\nP-value: {:.8f}\nR²: {:.2f}'.format(pearson_corr, p_value, r_squared),
         verticalalignment='top', horizontalalignment='left')

# Add labels to each point based on the index
for index, _ in df8.iterrows():
    plt.text(df8.loc[index, 'Score-R 3-BL'], df8.loc[index, 'Aß 1-42 3-BL'], str(index), fontsize=8)

plt.title('Scatter Plot')
plt.xlabel('Score-R 3-BL')
plt.ylabel('Aß 1-42 3-BL')

plt.savefig('df8.png')

# Show plot
plt.show()

In [ ]:
df9 = pd.read_excel("correlation.xlsx", sheet_name=8, index_col=0)  # 0 refers to the first sheet
df9.fillna(0, inplace=True)
df9

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import pearsonr

# Assuming df1 is your DataFrame containing the data

# Plot the scatter plot with regression line and rho value
plt.figure(figsize=(8, 6))
sns.regplot(x=df9['Score-R 6-3'], y=df9['Aß 1-40 6-3'], scatter_kws={'alpha':0.5}, line_kws={'color': 'black'}, color='blue')

# Calculate Pearson correlation coefficient and p-value
pearson_corr, p_value = pearsonr(df9['Score-R 6-3'], df9['Aß 1-40 6-3'])

# Calculate R-squared value
r_squared = pearson_corr**2

# Add Pearson correlation coefficient, p-value, and R-squared value as text
plt.text(df9['Score-R 6-3'].min(), df9['Aß 1-40 6-3'].max(), 
         'Pearson Correlation: {:.2f}\nP-value: {:.8f}\nR²: {:.2f}'.format(pearson_corr, p_value, r_squared),
         verticalalignment='bottom', horizontalalignment='left')

# Add labels to each point based on the index
for index, _ in df9.iterrows():
    plt.text(df9.loc[index, 'Score-R 6-3'], df9.loc[index, 'Aß 1-40 6-3'], str(index), fontsize=8)

plt.title('Scatter Plot')
plt.xlabel('Score-R 6-3')
plt.ylabel('Aß 1-40 6-3')

plt.savefig('df9.png')

# Show plot
plt.show()

In [ ]:
df10 = pd.read_excel("correlation.xlsx", sheet_name=9, index_col=0)  # 0 refers to the first sheet
df10.fillna(0, inplace=True)
df10

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import pearsonr

# Assuming df1 is your DataFrame containing the data

# Plot the scatter plot with regression line and rho value
plt.figure(figsize=(8, 6))
sns.regplot(x=df10['Score-R 6-3'], y=df10['Aß 1-42 6-3'], scatter_kws={'alpha':0.5}, line_kws={'color': 'black'}, color='blue')

# Calculate Pearson correlation coefficient and p-value
pearson_corr, p_value = pearsonr(df10['Score-R 6-3'], df10['Aß 1-42 6-3'])

# Calculate R-squared value
r_squared = pearson_corr**2

# Add Pearson correlation coefficient, p-value, and R-squared value as text
plt.text(df10['Score-R 6-3'].min(), df10['Aß 1-42 6-3'].max(), 
         'Pearson Correlation: {:.2f}\nP-value: {:.8f}\nR²: {:.2f}'.format(pearson_corr, p_value, r_squared),
         verticalalignment='bottom', horizontalalignment='left')

# Add labels to each point based on the index
for index, _ in df10.iterrows():
    plt.text(df10.loc[index, 'Score-R 6-3'], df10.loc[index, 'Aß 1-42 6-3'], str(index), fontsize=8)

plt.title('Scatter Plot')
plt.xlabel('Score-R 6-3')
plt.ylabel('Aß 1-42 6-3')

plt.savefig('df10.png')

# Show plot
plt.show()

In [ ]:
df11 = pd.read_excel("correlation_highNfH_Aß42.xlsx", sheet_name=0, index_col=0)  # 0 refers to the first sheet
df11.fillna(0, inplace=True)
df11

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import pearsonr

# Assuming df1 is your DataFrame containing the data

# Plot the scatter plot with regression line and rho value
plt.figure(figsize=(8, 6))
sns.regplot(x=df11['pNfH 6-BL'], y=df11['Aß 1-42 6-BL'], scatter_kws={'alpha':0.5}, line_kws={'color': 'black'}, color='blue')

# Calculate Pearson correlation coefficient and p-value
pearson_corr, p_value = pearsonr(df11['pNfH 6-BL'], df11['Aß 1-42 6-BL'])

# Calculate R-squared value
r_squared = pearson_corr**2

# Add Pearson correlation coefficient, p-value, and R-squared value as text
plt.text(df11['pNfH 6-BL'].min(), df11['Aß 1-42 6-BL'].max(), 
         'Pearson Correlation: {:.2f}\nP-value: {:.8f}\nR²: {:.2f}'.format(pearson_corr, p_value, r_squared),
         verticalalignment='bottom', horizontalalignment='left')

# Add labels to each point based on the index
for index, _ in df11.iterrows():
    plt.text(df11.loc[index, 'pNfH 6-BL'], df11.loc[index, 'Aß 1-42 6-BL'], str(index), fontsize=8)

plt.title('Scatter Plot')
plt.xlabel('pNfH 6-BL')
plt.ylabel('Aß 1-42 6-BL')

plt.savefig('df11.png')

# Show plot
plt.show()

In [ ]:
df12 = pd.read_excel("correlation_highNfH_Aß42.xlsx", sheet_name=1, index_col=0)  # 0 refers to the first sheet
df12.fillna(0, inplace=True)
df12

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import pearsonr

# Assuming df1 is your DataFrame containing the data

# Plot the scatter plot with regression line and rho value
plt.figure(figsize=(8, 6))
sns.regplot(x=df12['pNfH 3-BL'], y=df12['Aß 1-42 3-BL'], scatter_kws={'alpha':0.5}, line_kws={'color': 'black'}, color='blue')

# Calculate Pearson correlation coefficient and p-value
pearson_corr, p_value = pearsonr(df12['pNfH 3-BL'], df12['Aß 1-42 3-BL'])

# Calculate R-squared value
r_squared = pearson_corr**2

# Add Pearson correlation coefficient, p-value, and R-squared value as text
plt.text(df12['pNfH 3-BL'].min(), df12['Aß 1-42 3-BL'].max(), 
         'Pearson Correlation: {:.2f}\nP-value: {:.8f}\nR²: {:.2f}'.format(pearson_corr, p_value, r_squared),
         verticalalignment='bottom', horizontalalignment='left')

# Add labels to each point based on the index
for index, _ in df12.iterrows():
    plt.text(df12.loc[index, 'pNfH 3-BL'], df12.loc[index, 'Aß 1-42 3-BL'], str(index), fontsize=8)

plt.title('Scatter Plot')
plt.xlabel('pNfH 3-BL')
plt.ylabel('Aß 1-42 3-BL')

plt.savefig('df12.png')

# Show plot
plt.show()

In [ ]:
df_new_1 = pd.read_excel("correlation281124.xlsx", sheet_name=0, index_col=0)  # 0 refers to the first sheet
df_new_1.fillna(0, inplace=True)
df_new_1

In [ ]:
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import pearsonr, f_oneway, t
from sklearn.linear_model import LinearRegression
from adjustText import adjust_text

# Bootstrapping function for Pearson correlation coefficient
def bootstrap_pearsonr(data_x, data_y, n_bootstraps=1000, ci=95):
    bootstrapped_corrs = []
    for _ in range(n_bootstraps):
        sample_indices = np.random.choice(len(data_x), len(data_x), replace=True)
        sample_x = data_x[sample_indices]
        sample_y = data_y[sample_indices]
        boot_corr, _ = pearsonr(sample_x, sample_y)
        bootstrapped_corrs.append(boot_corr)
    lower_bound = np.percentile(bootstrapped_corrs, (100 - ci) / 2)
    upper_bound = np.percentile(bootstrapped_corrs, 100 - (100 - ci) / 2)
    return lower_bound, upper_bound

# Assuming df_new_1 is your DataFrame containing the data
data_x = df_new_1['pNfH 6-BL'].values.reshape(-1, 1)
data_y = df_new_1['UCHL1 6-BL'].values

# Linear regression model for adjusted R²
model = LinearRegression()
model.fit(data_x, data_y)
r_squared = model.score(data_x, data_y)
n = len(data_x)  # Number of samples
p = 1  # Number of predictors
adjusted_r_squared = 1 - ((1 - r_squared) * (n - 1) / (n - p - 1))

# F-test calculation
f_statistic, f_p_value = f_oneway(data_x.flatten(), data_y)

# Calculate Pearson correlation coefficient and p-value
pearson_corr, p_value = pearsonr(data_x.flatten(), data_y)

# Calculate confidence intervals
ci_lower, ci_upper = bootstrap_pearsonr(data_x.flatten(), data_y)

# Plot the regression line without scatter points and custom confidence interval color
plt.figure(figsize=(8, 8))
ax = sns.regplot(
    x=data_x.flatten(),
    y=data_y,
    scatter=False,  # Disable scatter points from regplot
    line_kws={'linestyle': '--', 'color': 'black'},
    ci=None,  # Disable automatic confidence interval shading
    color='MediumVioletRed'
)

# Generate x-values for prediction
x_vals = np.linspace(data_x.min(), data_x.max(), 100)
y_pred = model.predict(x_vals.reshape(-1, 1))

# Calculate the standard error of the regression
confidence = 0.95
t_value = t.ppf((1 + confidence) / 2., n - 2)
s_err = np.sqrt(np.sum((data_y - model.predict(data_x)) ** 2) / (n - 2))
y_err = t_value * s_err * np.sqrt(1 / n + (x_vals - np.mean(data_x)) ** 2 / np.sum((data_x - np.mean(data_x)) ** 2))

# Confidence interval shading with a new color
ci_upper_shade = y_pred + y_err
ci_lower_shade = y_pred - y_err

plt.fill_between(
    x_vals,
    ci_lower_shade,
    ci_upper_shade,
    color='MediumVioletRed',  # Change the grey to light blue or any color
    alpha=0.15  # Adjust transparency
)

# Add the scatter points
plt.scatter(
    data_x.flatten(),
    data_y,
    alpha=1,
    s=75,  # Size of markers
    marker='^',  # Custom marker shape
    color='MediumVioletRed'
)

# Add text annotations for points
text_annotations = []
x_offset = 0  # Horizontal offset
y_offset = 25   # Vertical offset

for index, _ in df_new_1.iterrows():
    text = plt.text(
        df_new_1.loc[index, 'pNfH 6-BL'] + x_offset,  # Initial x position with offset
        df_new_1.loc[index, 'UCHL1 6-BL'] + y_offset,  # Initial y position with offset
        str(index),
        fontsize=10,
        fontweight='bold'  # Make text bold
    )
    text_annotations.append(text)  # Add the text object to the list

# Automatically adjust the position of text to avoid overlaps
adjust_text(
    text_annotations,
    arrowprops=dict(arrowstyle='->', color='black', lw=2.5)  # Optional: Add arrows to point back to scatter points
)

# Add Pearson correlation coefficient, p-value, R², adjusted R², F-statistic, and CI as a table
table_data = [
    ['Pearson Correlation', f'{pearson_corr:.2f}'],
    ['P-value', f'{p_value:.8f}'],
    ['R²', f'{r_squared:.2f}'],
    ['Adjusted R²', f'{adjusted_r_squared:.2f}'],
    ['F-statistic', f'{f_statistic:.2f}'],
    ['F-test P-value', f'{f_p_value:.8f}'],
    ['95% CI', f'[{ci_lower:.2f}, {ci_upper:.2f}]']
]

# Add the table below the plot
plt.table(cellText=table_data,
          colLabels=['Statistic', 'Value'],
          cellLoc='center', 
          loc='bottom', 
          bbox=[0.0, -0.4, 1, 0.3])  # Adjust bbox to position the table

plt.title('Scatter Plot')
plt.xlabel('pNfH 6-BL')
plt.ylabel('UCHL1 6-BL')

plt.tight_layout()
plt.savefig('df_new_1.png', dpi=300)

# Show plot
plt.show()


In [ ]:
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import pearsonr, f_oneway, t
from sklearn.linear_model import LinearRegression
from adjustText import adjust_text

# Bootstrapping function for Pearson correlation coefficient
def bootstrap_pearsonr(data_x, data_y, n_bootstraps=1000, ci=95):
    bootstrapped_corrs = []
    for _ in range(n_bootstraps):
        sample_indices = np.random.choice(len(data_x), len(data_x), replace=True)
        sample_x = data_x[sample_indices]
        sample_y = data_y[sample_indices]
        boot_corr, _ = pearsonr(sample_x, sample_y)
        bootstrapped_corrs.append(boot_corr)
    lower_bound = np.percentile(bootstrapped_corrs, (100 - ci) / 2)
    upper_bound = np.percentile(bootstrapped_corrs, 100 - (100 - ci) / 2)
    return lower_bound, upper_bound

# Assuming df_new_1 is your DataFrame containing the data
data_x = df_new_1['pNfH 6-BL'].values.reshape(-1, 1)
data_y = df_new_1['UCHL1 6-BL'].values

# Linear regression model for adjusted R²
model = LinearRegression()
model.fit(data_x, data_y)
r_squared = model.score(data_x, data_y)
n = len(data_x)  # Number of samples
p = 1  # Number of predictors
adjusted_r_squared = 1 - ((1 - r_squared) * (n - 1) / (n - p - 1))

# F-test calculation
f_statistic, f_p_value = f_oneway(data_x.flatten(), data_y)

# Calculate Pearson correlation coefficient and p-value
pearson_corr, p_value = pearsonr(data_x.flatten(), data_y)

# Calculate confidence intervals
ci_lower, ci_upper = bootstrap_pearsonr(data_x.flatten(), data_y)

# Plot the regression line without scatter points and custom confidence interval color
plt.figure(figsize=(8, 8))
ax = sns.regplot(
    x=data_x.flatten(),
    y=data_y,
    scatter=False,  # Disable scatter points from regplot
    line_kws={'linestyle': '--', 'color': 'black'},
    ci=None,  # Disable automatic confidence interval shading
    color='MediumVioletRed'
)

# Generate x-values for prediction
x_vals = np.linspace(data_x.min(), data_x.max(), 100)
y_pred = model.predict(x_vals.reshape(-1, 1))

# Calculate the standard error of the regression
confidence = 0.95
t_value = t.ppf((1 + confidence) / 2., n - 2)
s_err = np.sqrt(np.sum((data_y - model.predict(data_x)) ** 2) / (n - 2))
y_err = t_value * s_err * np.sqrt(1 / n + (x_vals - np.mean(data_x)) ** 2 / np.sum((data_x - np.mean(data_x)) ** 2))

# Confidence interval shading with a new color
ci_upper_shade = y_pred + y_err
ci_lower_shade = y_pred - y_err

plt.fill_between(
    x_vals,
    ci_lower_shade,
    ci_upper_shade,
    color='MediumVioletRed',  # Change the grey to light blue or any color
    alpha=0.15  # Adjust transparency
)

# Add the scatter points
plt.scatter(
    data_x.flatten(),
    data_y,
    alpha=1,
    s=75,  # Size of markers
    marker='^',  # Custom marker shape
    color='MediumVioletRed'
)

# Add Pearson correlation coefficient, p-value, R², adjusted R², F-statistic, and CI as a table
table_data = [
    ['Pearson Correlation', f'{pearson_corr:.2f}'],
    ['P-value', f'{p_value:.8f}'],
    ['R²', f'{r_squared:.2f}'],
    ['Adjusted R²', f'{adjusted_r_squared:.2f}'],
    ['F-statistic', f'{f_statistic:.2f}'],
    ['F-test P-value', f'{f_p_value:.8f}'],
    ['95% CI', f'[{ci_lower:.2f}, {ci_upper:.2f}]']
]

# Add the table below the plot
plt.table(cellText=table_data,
          colLabels=['Statistic', 'Value'],
          cellLoc='center', 
          loc='bottom', 
          bbox=[0.0, -0.4, 1, 0.3])  # Adjust bbox to position the table

plt.title('Scatter Plot')
plt.xlabel('pNfH 6-BL')
plt.ylabel('UCHL1 6-BL')

plt.tight_layout()
plt.savefig('df_new_1_wo_labels.png', dpi=300)

# Show plot
plt.show()


In [ ]:
df_new_2 = pd.read_excel("correlation281124.xlsx", sheet_name=1, index_col=0)  # 0 refers to the first sheet
df_new_2.fillna(0, inplace=True)
df_new_2

In [ ]:
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import pearsonr, f_oneway, t
from sklearn.linear_model import LinearRegression
from adjustText import adjust_text

# Bootstrapping function for Pearson correlation coefficient
def bootstrap_pearsonr(data_x, data_y, n_bootstraps=1000, ci=95):
    bootstrapped_corrs = []
    for _ in range(n_bootstraps):
        sample_indices = np.random.choice(len(data_x), len(data_x), replace=True)
        sample_x = data_x[sample_indices]
        sample_y = data_y[sample_indices]
        boot_corr, _ = pearsonr(sample_x, sample_y)
        bootstrapped_corrs.append(boot_corr)
    lower_bound = np.percentile(bootstrapped_corrs, (100 - ci) / 2)
    upper_bound = np.percentile(bootstrapped_corrs, 100 - (100 - ci) / 2)
    return lower_bound, upper_bound

# Assuming df_new_2 is your DataFrame containing the data
data_x = df_new_2['NfL 6-BL'].values.reshape(-1, 1)
data_y = df_new_2['UCHL1 6-BL'].values

# Linear regression model for adjusted R²
model = LinearRegression()
model.fit(data_x, data_y)
r_squared = model.score(data_x, data_y)
n = len(data_x)  # Number of samples
p = 1  # Number of predictors
adjusted_r_squared = 1 - ((1 - r_squared) * (n - 1) / (n - p - 1))

# F-test calculation
f_statistic, f_p_value = f_oneway(data_x.flatten(), data_y)

# Calculate Pearson correlation coefficient and p-value
pearson_corr, p_value = pearsonr(data_x.flatten(), data_y)

# Calculate confidence intervals
ci_lower, ci_upper = bootstrap_pearsonr(data_x.flatten(), data_y)

# Plot the regression line without scatter points and custom confidence interval color
plt.figure(figsize=(8, 8))
ax = sns.regplot(
    x=data_x.flatten(),
    y=data_y,
    scatter=False,  # Disable scatter points from regplot
    line_kws={'linestyle': '--', 'color': 'black'},
    ci=None,  # Disable automatic confidence interval shading
    color='blue'
)

# Generate x-values for prediction
x_vals = np.linspace(data_x.min(), data_x.max(), 100)
y_pred = model.predict(x_vals.reshape(-1, 1))

# Calculate the standard error of the regression
confidence = 0.95
t_value = t.ppf((1 + confidence) / 2., n - 2)
s_err = np.sqrt(np.sum((data_y - model.predict(data_x)) ** 2) / (n - 2))
y_err = t_value * s_err * np.sqrt(1 / n + (x_vals - np.mean(data_x)) ** 2 / np.sum((data_x - np.mean(data_x)) ** 2))

# Confidence interval shading
ci_upper_shade = y_pred + y_err
ci_lower_shade = y_pred - y_err

plt.fill_between(
    x_vals,
    ci_lower_shade,
    ci_upper_shade,
    color='MediumVioletRed',
    alpha=0.15  # Adjust transparency
)

# Add the scatter points
plt.scatter(
    data_x.flatten(),
    data_y,
    alpha=1,
    s=75,  # Size of markers
    marker='^',  # Custom marker shape
    color='MediumVioletRed'
)

# Add text annotations for points
text_annotations = []
x_offset = 0  # Horizontal offset
y_offset = 25   # Vertical offset

for index, _ in df_new_2.iterrows():
    text = plt.text(
        df_new_2.loc[index, 'NfL 6-BL'] + x_offset,  # Initial x position with offset
        df_new_2.loc[index, 'UCHL1 6-BL'] + y_offset,  # Initial y position with offset
        str(index),
        fontsize=10,
        fontweight='bold'  # Make text bold
    )
    text_annotations.append(text)  # Add the text object to the list

# Automatically adjust the position of text to avoid overlaps
adjust_text(
    text_annotations,
    arrowprops=dict(arrowstyle='->', color='black', lw=2.5)  # Optional: Add arrows to point back to scatter points
)

# Add Pearson correlation coefficient, p-value, R², adjusted R², F-statistic, and CI as a table
table_data = [
    ['Pearson Correlation', f'{pearson_corr:.2f}'],
    ['P-value', f'{p_value:.8f}'],
    ['R²', f'{r_squared:.2f}'],
    ['Adjusted R²', f'{adjusted_r_squared:.2f}'],
    ['F-statistic', f'{f_statistic:.2f}'],
    ['F-test P-value', f'{f_p_value:.8f}'],
    ['95% CI', f'[{ci_lower:.2f}, {ci_upper:.2f}]']
]

# Add the table below the plot
plt.table(cellText=table_data,
          colLabels=['Statistic', 'Value'],
          cellLoc='center', 
          loc='bottom', 
          bbox=[0.0, -0.4, 1, 0.3])  # Adjust bbox to position the table

plt.title('Scatter Plot with Statistics')
plt.xlabel('NfL 6-BL')
plt.ylabel('UCHL1 6-BL')

plt.tight_layout()
plt.savefig('df_new_2.png', dpi=300)

# Show plot
plt.show()


In [ ]:
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import pearsonr, f_oneway, t
from sklearn.linear_model import LinearRegression
from adjustText import adjust_text

# Bootstrapping function for Pearson correlation coefficient
def bootstrap_pearsonr(data_x, data_y, n_bootstraps=1000, ci=95):
    bootstrapped_corrs = []
    for _ in range(n_bootstraps):
        sample_indices = np.random.choice(len(data_x), len(data_x), replace=True)
        sample_x = data_x[sample_indices]
        sample_y = data_y[sample_indices]
        boot_corr, _ = pearsonr(sample_x, sample_y)
        bootstrapped_corrs.append(boot_corr)
    lower_bound = np.percentile(bootstrapped_corrs, (100 - ci) / 2)
    upper_bound = np.percentile(bootstrapped_corrs, 100 - (100 - ci) / 2)
    return lower_bound, upper_bound

# Assuming df_new_2 is your DataFrame containing the data
data_x = df_new_2['NfL 6-BL'].values.reshape(-1, 1)
data_y = df_new_2['UCHL1 6-BL'].values

# Linear regression model for adjusted R²
model = LinearRegression()
model.fit(data_x, data_y)
r_squared = model.score(data_x, data_y)
n = len(data_x)  # Number of samples
p = 1  # Number of predictors
adjusted_r_squared = 1 - ((1 - r_squared) * (n - 1) / (n - p - 1))

# F-test calculation
f_statistic, f_p_value = f_oneway(data_x.flatten(), data_y)

# Calculate Pearson correlation coefficient and p-value
pearson_corr, p_value = pearsonr(data_x.flatten(), data_y)

# Calculate confidence intervals
ci_lower, ci_upper = bootstrap_pearsonr(data_x.flatten(), data_y)

# Plot the regression line without scatter points and custom confidence interval color
plt.figure(figsize=(8, 8))
ax = sns.regplot(
    x=data_x.flatten(),
    y=data_y,
    scatter=False,  # Disable scatter points from regplot
    line_kws={'linestyle': '--', 'color': 'black'},
    ci=None,  # Disable automatic confidence interval shading
    color='blue'
)

# Generate x-values for prediction
x_vals = np.linspace(data_x.min(), data_x.max(), 100)
y_pred = model.predict(x_vals.reshape(-1, 1))

# Calculate the standard error of the regression
confidence = 0.95
t_value = t.ppf((1 + confidence) / 2., n - 2)
s_err = np.sqrt(np.sum((data_y - model.predict(data_x)) ** 2) / (n - 2))
y_err = t_value * s_err * np.sqrt(1 / n + (x_vals - np.mean(data_x)) ** 2 / np.sum((data_x - np.mean(data_x)) ** 2))

# Confidence interval shading
ci_upper_shade = y_pred + y_err
ci_lower_shade = y_pred - y_err

plt.fill_between(
    x_vals,
    ci_lower_shade,
    ci_upper_shade,
    color='MediumVioletRed',
    alpha=0.15  # Adjust transparency
)

# Add the scatter points
plt.scatter(
    data_x.flatten(),
    data_y,
    alpha=1,
    s=75,  # Size of markers
    marker='^',  # Custom marker shape
    color='MediumVioletRed'
)

# Add Pearson correlation coefficient, p-value, R², adjusted R², F-statistic, and CI as a table
table_data = [
    ['Pearson Correlation', f'{pearson_corr:.2f}'],
    ['P-value', f'{p_value:.8f}'],
    ['R²', f'{r_squared:.2f}'],
    ['Adjusted R²', f'{adjusted_r_squared:.2f}'],
    ['F-statistic', f'{f_statistic:.2f}'],
    ['F-test P-value', f'{f_p_value:.8f}'],
    ['95% CI', f'[{ci_lower:.2f}, {ci_upper:.2f}]']
]

# Add the table below the plot
plt.table(cellText=table_data,
          colLabels=['Statistic', 'Value'],
          cellLoc='center', 
          loc='bottom', 
          bbox=[0.0, -0.4, 1, 0.3])  # Adjust bbox to position the table

plt.title('Scatter Plot with Statistics')
plt.xlabel('NfL 6-BL')
plt.ylabel('UCHL1 6-BL')

plt.tight_layout()
plt.savefig('df_new_2_wo_labels.png', dpi=300)

# Show plot
plt.show()


In [ ]:
df_new_3 = pd.read_excel("correlation281124.xlsx", sheet_name=2, index_col=0)  # 0 refers to the first sheet
df_new_3.fillna(0, inplace=True)
df_new_3

In [ ]:
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import pearsonr, t
from sklearn.linear_model import LinearRegression
from adjustText import adjust_text

# Assuming df_new_3 is your DataFrame containing the data
data_x = df_new_3['pNfH 6-BL'].values.reshape(-1, 1)
data_y = df_new_3['Aß 1-40 6-BL'].values

# Linear regression model
model = LinearRegression()
model.fit(data_x, data_y)
r_squared = model.score(data_x, data_y)
pearson_corr, p_value = pearsonr(data_x.flatten(), data_y)

# Calculate adjusted R-squared
n = len(data_x)  # Number of samples
p = 1  # Number of predictors
adjusted_r_squared = 1 - ((1 - r_squared) * (n - 1) / (n - p - 1))

# Bootstrap function for Pearson correlation confidence intervals
def bootstrap_pearsonr(data_x, data_y, n_bootstraps=1000, ci=95):
    bootstrapped_corrs = []
    for _ in range(n_bootstraps):
        sample_indices = np.random.choice(len(data_x), len(data_x), replace=True)
        sample_x = data_x[sample_indices]
        sample_y = data_y[sample_indices]
        boot_corr, _ = pearsonr(sample_x, sample_y)
        bootstrapped_corrs.append(boot_corr)
    lower_bound = np.percentile(bootstrapped_corrs, (100 - ci) / 2)
    upper_bound = np.percentile(bootstrapped_corrs, 100 - (100 - ci) / 2)
    return lower_bound, upper_bound

ci_lower, ci_upper = bootstrap_pearsonr(data_x.flatten(), data_y)

# Plot scatter and regression line
plt.figure(figsize=(8, 8))
sns.regplot(
    x=data_x.flatten(),
    y=data_y,
    scatter=False,
    line_kws={'color': 'black', 'linestyle': '--'},
    color='blue',
    ci=None
)

# Generate x-values for prediction and calculate confidence intervals
x_vals = np.linspace(data_x.min(), data_x.max(), 100)
y_pred = model.predict(x_vals.reshape(-1, 1))

confidence = 0.95
t_value = t.ppf((1 + confidence) / 2., n - 2)
s_err = np.sqrt(np.sum((data_y - model.predict(data_x)) ** 2) / (n - 2))
y_err = t_value * s_err * np.sqrt(1 / n + (x_vals - np.mean(data_x)) ** 2 / np.sum((data_x - np.mean(data_x)) ** 2))

# Confidence interval shading
plt.fill_between(
    x_vals,
    y_pred - y_err,
    y_pred + y_err,
    color='MediumVioletRed',
    alpha=0.15
)

# Add scatter points
plt.scatter(
    data_x.flatten(),
    data_y,
    alpha=1,
    s=75,
    marker='^',  # Custom marker shape
    color='MediumVioletRed'
)

# Add text annotations for points
text_annotations = []
for index, _ in df_new_3.iterrows():
    text = plt.text(
        df_new_3.loc[index, 'pNfH 6-BL'],
        df_new_3.loc[index, 'Aß 1-40 6-BL'],
        str(index),
        fontsize=10,
        fontweight='bold'
    )
    text_annotations.append(text)

# Automatically adjust text positions
adjust_text(
    text_annotations,
    arrowprops=dict(arrowstyle='->', color='black', lw=1.5)
)

# Add statistics as a table
table_data = [
    ['Pearson Correlation', f'{pearson_corr:.2f}'],
    ['P-value', f'{p_value:.8f}'],
    ['R²', f'{r_squared:.2f}'],
    ['Adjusted R²', f'{adjusted_r_squared:.2f}'],
    ['95% CI', f'[{ci_lower:.2f}, {ci_upper:.2f}]']
]

plt.table(
    cellText=table_data,
    colLabels=['Statistic', 'Value'],
    cellLoc='center',
    loc='bottom',
    bbox=[0.0, -0.4, 1, 0.3]
)

plt.title('Scatter Plot with Regression and Statistics')
plt.xlabel('pNfH 6-BL')
plt.ylabel('Aß 1-40 6-BL')

plt.tight_layout()
plt.savefig('df_new_3.png', dpi=300)
plt.show()

In [ ]:
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import pearsonr, t
from sklearn.linear_model import LinearRegression
from adjustText import adjust_text

# Assuming df_new_3 is your DataFrame containing the data
data_x = df_new_3['pNfH 6-BL'].values.reshape(-1, 1)
data_y = df_new_3['Aß 1-40 6-BL'].values

# Linear regression model
model = LinearRegression()
model.fit(data_x, data_y)
r_squared = model.score(data_x, data_y)
pearson_corr, p_value = pearsonr(data_x.flatten(), data_y)

# Calculate adjusted R-squared
n = len(data_x)  # Number of samples
p = 1  # Number of predictors
adjusted_r_squared = 1 - ((1 - r_squared) * (n - 1) / (n - p - 1))

# Bootstrap function for Pearson correlation confidence intervals
def bootstrap_pearsonr(data_x, data_y, n_bootstraps=1000, ci=95):
    bootstrapped_corrs = []
    for _ in range(n_bootstraps):
        sample_indices = np.random.choice(len(data_x), len(data_x), replace=True)
        sample_x = data_x[sample_indices]
        sample_y = data_y[sample_indices]
        boot_corr, _ = pearsonr(sample_x, sample_y)
        bootstrapped_corrs.append(boot_corr)
    lower_bound = np.percentile(bootstrapped_corrs, (100 - ci) / 2)
    upper_bound = np.percentile(bootstrapped_corrs, 100 - (100 - ci) / 2)
    return lower_bound, upper_bound

ci_lower, ci_upper = bootstrap_pearsonr(data_x.flatten(), data_y)

# Plot scatter and regression line
plt.figure(figsize=(8, 8))
sns.regplot(
    x=data_x.flatten(),
    y=data_y,
    scatter=False,
    line_kws={'color': 'black', 'linestyle': '--'},
    color='blue',
    ci=None
)

# Generate x-values for prediction and calculate confidence intervals
x_vals = np.linspace(data_x.min(), data_x.max(), 100)
y_pred = model.predict(x_vals.reshape(-1, 1))

confidence = 0.95
t_value = t.ppf((1 + confidence) / 2., n - 2)
s_err = np.sqrt(np.sum((data_y - model.predict(data_x)) ** 2) / (n - 2))
y_err = t_value * s_err * np.sqrt(1 / n + (x_vals - np.mean(data_x)) ** 2 / np.sum((data_x - np.mean(data_x)) ** 2))

# Confidence interval shading
plt.fill_between(
    x_vals,
    y_pred - y_err,
    y_pred + y_err,
    color='MediumVioletRed',
    alpha=0.15
)

# Add scatter points
plt.scatter(
    data_x.flatten(),
    data_y,
    alpha=1,
    s=75,
    marker='^',  # Custom marker shape
    color='MediumVioletRed'
)


# Add statistics as a table
table_data = [
    ['Pearson Correlation', f'{pearson_corr:.2f}'],
    ['P-value', f'{p_value:.8f}'],
    ['R²', f'{r_squared:.2f}'],
    ['Adjusted R²', f'{adjusted_r_squared:.2f}'],
    ['95% CI', f'[{ci_lower:.2f}, {ci_upper:.2f}]']
]

plt.table(
    cellText=table_data,
    colLabels=['Statistic', 'Value'],
    cellLoc='center',
    loc='bottom',
    bbox=[0.0, -0.4, 1, 0.3]
)

plt.title('Scatter Plot with Regression and Statistics')
plt.xlabel('pNfH 6-BL')
plt.ylabel('Aß 1-40 6-BL')

plt.tight_layout()
plt.savefig('df_new_3_wo_labels.png', dpi=300)
plt.show()

In [ ]:
df_new_4 = pd.read_excel("correlation281124.xlsx", sheet_name=3, index_col=0)  # 0 refers to the first sheet
df_new_4.fillna(0, inplace=True)
df_new_4

In [ ]:
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import pearsonr, t
from sklearn.linear_model import LinearRegression
from adjustText import adjust_text

# Assuming df_new_4 is your DataFrame containing the data
data_x = df_new_4['NfL 6-BL'].values.reshape(-1, 1)
data_y = df_new_4['Aß 1-40 6-BL'].values

# Linear regression model
model = LinearRegression()
model.fit(data_x, data_y)
r_squared = model.score(data_x, data_y)
pearson_corr, p_value = pearsonr(data_x.flatten(), data_y)

# Calculate adjusted R-squared
n = len(data_x)  # Number of samples
p = 1  # Number of predictors
adjusted_r_squared = 1 - ((1 - r_squared) * (n - 1) / (n - p - 1))

# Bootstrap function for Pearson correlation confidence intervals
def bootstrap_pearsonr(data_x, data_y, n_bootstraps=1000, ci=95):
    bootstrapped_corrs = []
    for _ in range(n_bootstraps):
        sample_indices = np.random.choice(len(data_x), len(data_x), replace=True)
        sample_x = data_x[sample_indices]
        sample_y = data_y[sample_indices]
        boot_corr, _ = pearsonr(sample_x, sample_y)
        bootstrapped_corrs.append(boot_corr)
    lower_bound = np.percentile(bootstrapped_corrs, (100 - ci) / 2)
    upper_bound = np.percentile(bootstrapped_corrs, 100 - (100 - ci) / 2)
    return lower_bound, upper_bound

ci_lower, ci_upper = bootstrap_pearsonr(data_x.flatten(), data_y)

# Plot scatter and regression line
plt.figure(figsize=(8, 8))
sns.regplot(
    x=data_x.flatten(),
    y=data_y,
    scatter=False,
    line_kws={'color': 'black', 'linestyle': '--'},
    color='MediumVioletRed',
    ci=None
)

# Generate x-values for prediction and calculate confidence intervals
x_vals = np.linspace(data_x.min(), data_x.max(), 100)
y_pred = model.predict(x_vals.reshape(-1, 1))

confidence = 0.95
t_value = t.ppf((1 + confidence) / 2., n - 2)
s_err = np.sqrt(np.sum((data_y - model.predict(data_x)) ** 2) / (n - 2))
y_err = t_value * s_err * np.sqrt(1 / n + (x_vals - np.mean(data_x)) ** 2 / np.sum((data_x - np.mean(data_x)) ** 2))

# Confidence interval shading
plt.fill_between(
    x_vals,
    y_pred - y_err,
    y_pred + y_err,
    color='MediumVioletRed',
    alpha=0.15
)

# Add scatter points
plt.scatter(
    data_x.flatten(),
    data_y,
    alpha=1,
    s=75,
    marker='^',  # Custom marker shape
    color='MediumVioletRed'
)

# Add text annotations for points
text_annotations = []
x_offset = 0  # Horizontal offset
y_offset = 25   # Vertical offset

for index, _ in df_new_4.iterrows():
    text = plt.text(
        df_new_4.loc[index, 'NfL 6-BL'] + x_offset,
        df_new_4.loc[index, 'Aß 1-40 6-BL'] + y_offset,
        str(index),
        fontsize=10,
        fontweight='bold'
    )
    text_annotations.append(text)

# Automatically adjust text positions
adjust_text(
    text_annotations,
    arrowprops=dict(arrowstyle='->', color='black', lw=1.5)
)

# Add statistics as a table
table_data = [
    ['Pearson Correlation', f'{pearson_corr:.2f}'],
    ['P-value', f'{p_value:.8f}'],
    ['R²', f'{r_squared:.2f}'],
    ['Adjusted R²', f'{adjusted_r_squared:.2f}'],
    ['95% CI', f'[{ci_lower:.2f}, {ci_upper:.2f}]']
]

plt.table(
    cellText=table_data,
    colLabels=['Statistic', 'Value'],
    cellLoc='center',
    loc='bottom',
    bbox=[0.0, -0.4, 1, 0.3]
)

plt.title('Scatter Plot with Regression and Statistics')
plt.xlabel('NfL 6-BL')
plt.ylabel('Aß 1-40 6-BL')

plt.tight_layout()
plt.savefig('df_new_4.png', dpi=300)
plt.show()

In [ ]:
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import pearsonr, t
from sklearn.linear_model import LinearRegression
from adjustText import adjust_text

# Assuming df_new_4 is your DataFrame containing the data
data_x = df_new_4['NfL 6-BL'].values.reshape(-1, 1)
data_y = df_new_4['Aß 1-40 6-BL'].values

# Linear regression model
model = LinearRegression()
model.fit(data_x, data_y)
r_squared = model.score(data_x, data_y)
pearson_corr, p_value = pearsonr(data_x.flatten(), data_y)

# Calculate adjusted R-squared
n = len(data_x)  # Number of samples
p = 1  # Number of predictors
adjusted_r_squared = 1 - ((1 - r_squared) * (n - 1) / (n - p - 1))

# Bootstrap function for Pearson correlation confidence intervals
def bootstrap_pearsonr(data_x, data_y, n_bootstraps=1000, ci=95):
    bootstrapped_corrs = []
    for _ in range(n_bootstraps):
        sample_indices = np.random.choice(len(data_x), len(data_x), replace=True)
        sample_x = data_x[sample_indices]
        sample_y = data_y[sample_indices]
        boot_corr, _ = pearsonr(sample_x, sample_y)
        bootstrapped_corrs.append(boot_corr)
    lower_bound = np.percentile(bootstrapped_corrs, (100 - ci) / 2)
    upper_bound = np.percentile(bootstrapped_corrs, 100 - (100 - ci) / 2)
    return lower_bound, upper_bound

ci_lower, ci_upper = bootstrap_pearsonr(data_x.flatten(), data_y)

# Plot scatter and regression line
plt.figure(figsize=(8, 8))
sns.regplot(
    x=data_x.flatten(),
    y=data_y,
    scatter=False,
    line_kws={'color': 'black', 'linestyle': '--'},
    color='MediumVioletRed',
    ci=None
)

# Generate x-values for prediction and calculate confidence intervals
x_vals = np.linspace(data_x.min(), data_x.max(), 100)
y_pred = model.predict(x_vals.reshape(-1, 1))

confidence = 0.95
t_value = t.ppf((1 + confidence) / 2., n - 2)
s_err = np.sqrt(np.sum((data_y - model.predict(data_x)) ** 2) / (n - 2))
y_err = t_value * s_err * np.sqrt(1 / n + (x_vals - np.mean(data_x)) ** 2 / np.sum((data_x - np.mean(data_x)) ** 2))

# Confidence interval shading
plt.fill_between(
    x_vals,
    y_pred - y_err,
    y_pred + y_err,
    color='MediumVioletRed',
    alpha=0.15
)

# Add scatter points
plt.scatter(
    data_x.flatten(),
    data_y,
    alpha=1,
    s=75,
    marker='^',  # Custom marker shape
    color='MediumVioletRed'
)

# Add statistics as a table
table_data = [
    ['Pearson Correlation', f'{pearson_corr:.2f}'],
    ['P-value', f'{p_value:.8f}'],
    ['R²', f'{r_squared:.2f}'],
    ['Adjusted R²', f'{adjusted_r_squared:.2f}'],
    ['95% CI', f'[{ci_lower:.2f}, {ci_upper:.2f}]']
]

plt.table(
    cellText=table_data,
    colLabels=['Statistic', 'Value'],
    cellLoc='center',
    loc='bottom',
    bbox=[0.0, -0.4, 1, 0.3]
)

plt.title('Scatter Plot with Regression and Statistics')
plt.xlabel('NfL 6-BL')
plt.ylabel('Aß 1-40 6-BL')

plt.tight_layout()
plt.savefig('df_new_4_wo_labels.png', dpi=300)
plt.show()

In [ ]:
df_new_5 = pd.read_excel("correlation281124.xlsx", sheet_name=4, index_col=0)  # 0 refers to the first sheet
df_new_5.fillna(0, inplace=True)
df_new_5

In [ ]:
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import pearsonr, t
from sklearn.linear_model import LinearRegression
from adjustText import adjust_text

# Assuming df_new_5 is your DataFrame containing the data
data_x = df_new_5['pNfH 6-BL'].values.reshape(-1, 1)
data_y = df_new_5['Aß 1-42 6-BL'].values

# Linear regression model
model = LinearRegression()
model.fit(data_x, data_y)
r_squared = model.score(data_x, data_y)
pearson_corr, p_value = pearsonr(data_x.flatten(), data_y)

# Calculate adjusted R-squared
n = len(data_x)  # Number of samples
p = 1  # Number of predictors
adjusted_r_squared = 1 - ((1 - r_squared) * (n - 1) / (n - p - 1))

# Bootstrap function for Pearson correlation confidence intervals
def bootstrap_pearsonr(data_x, data_y, n_bootstraps=1000, ci=95):
    bootstrapped_corrs = []
    for _ in range(n_bootstraps):
        sample_indices = np.random.choice(len(data_x), len(data_x), replace=True)
        sample_x = data_x[sample_indices]
        sample_y = data_y[sample_indices]
        boot_corr, _ = pearsonr(sample_x, sample_y)
        bootstrapped_corrs.append(boot_corr)
    lower_bound = np.percentile(bootstrapped_corrs, (100 - ci) / 2)
    upper_bound = np.percentile(bootstrapped_corrs, 100 - (100 - ci) / 2)
    return lower_bound, upper_bound

ci_lower, ci_upper = bootstrap_pearsonr(data_x.flatten(), data_y)

# Plot scatter and regression line
plt.figure(figsize=(8, 8))
sns.regplot(
    x=data_x.flatten(),
    y=data_y,
    scatter=False,
    line_kws={'color': 'black', 'linestyle': '--'},
    color='MediumVioletRed',
    ci=None
)

# Generate x-values for prediction and calculate confidence intervals
x_vals = np.linspace(data_x.min(), data_x.max(), 100)
y_pred = model.predict(x_vals.reshape(-1, 1))

confidence = 0.95
t_value = t.ppf((1 + confidence) / 2., n - 2)
s_err = np.sqrt(np.sum((data_y - model.predict(data_x)) ** 2) / (n - 2))
y_err = t_value * s_err * np.sqrt(1 / n + (x_vals - np.mean(data_x)) ** 2 / np.sum((data_x - np.mean(data_x)) ** 2))

# Confidence interval shading
plt.fill_between(
    x_vals,
    y_pred - y_err,
    y_pred + y_err,
    color='MediumVioletRed',
    alpha=0.15
)

# Add scatter points
plt.scatter(
    data_x.flatten(),
    data_y,
    alpha=1,
    s=75,
    marker='^',  # Custom marker shape
    color='MediumVioletRed'
)

# Add text annotations for points
text_annotations = []
x_offset = 0  # Horizontal offset
y_offset = 25   # Vertical offset

for index, _ in df_new_5.iterrows():
    text = plt.text(
        df_new_5.loc[index, 'pNfH 6-BL'] + x_offset,
        df_new_5.loc[index, 'Aß 1-42 6-BL'] + y_offset,
        str(index),
        fontsize=10,
        fontweight='bold'
    )
    text_annotations.append(text)

# Automatically adjust text positions
adjust_text(
    text_annotations,
    arrowprops=dict(arrowstyle='->', color='black', lw=1.5)
)

# Add statistics as a table
table_data = [
    ['Pearson Correlation', f'{pearson_corr:.2f}'],
    ['P-value', f'{p_value:.8f}'],
    ['R²', f'{r_squared:.2f}'],
    ['Adjusted R²', f'{adjusted_r_squared:.2f}'],
    ['95% CI', f'[{ci_lower:.2f}, {ci_upper:.2f}]']
]

plt.table(
    cellText=table_data,
    colLabels=['Statistic', 'Value'],
    cellLoc='center',
    loc='bottom',
    bbox=[0.0, -0.4, 1, 0.3]
)

plt.title('Scatter Plot with Regression and Statistics')
plt.xlabel('pNfH 6-BL')
plt.ylabel('Aß 1-42 6-BL')

plt.tight_layout()
plt.savefig('df_new_5.png', dpi=300)
plt.show()

In [ ]:
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import pearsonr, t
from sklearn.linear_model import LinearRegression
from adjustText import adjust_text

# Assuming df_new_5 is your DataFrame containing the data
data_x = df_new_5['pNfH 6-BL'].values.reshape(-1, 1)
data_y = df_new_5['Aß 1-42 6-BL'].values

# Linear regression model
model = LinearRegression()
model.fit(data_x, data_y)
r_squared = model.score(data_x, data_y)
pearson_corr, p_value = pearsonr(data_x.flatten(), data_y)

# Calculate adjusted R-squared
n = len(data_x)  # Number of samples
p = 1  # Number of predictors
adjusted_r_squared = 1 - ((1 - r_squared) * (n - 1) / (n - p - 1))

# Bootstrap function for Pearson correlation confidence intervals
def bootstrap_pearsonr(data_x, data_y, n_bootstraps=1000, ci=95):
    bootstrapped_corrs = []
    for _ in range(n_bootstraps):
        sample_indices = np.random.choice(len(data_x), len(data_x), replace=True)
        sample_x = data_x[sample_indices]
        sample_y = data_y[sample_indices]
        boot_corr, _ = pearsonr(sample_x, sample_y)
        bootstrapped_corrs.append(boot_corr)
    lower_bound = np.percentile(bootstrapped_corrs, (100 - ci) / 2)
    upper_bound = np.percentile(bootstrapped_corrs, 100 - (100 - ci) / 2)
    return lower_bound, upper_bound

ci_lower, ci_upper = bootstrap_pearsonr(data_x.flatten(), data_y)

# Plot scatter and regression line
plt.figure(figsize=(8, 8))
sns.regplot(
    x=data_x.flatten(),
    y=data_y,
    scatter=False,
    line_kws={'color': 'black', 'linestyle': '--'},
    color='MediumVioletRed',
    ci=None
)

# Generate x-values for prediction and calculate confidence intervals
x_vals = np.linspace(data_x.min(), data_x.max(), 100)
y_pred = model.predict(x_vals.reshape(-1, 1))

confidence = 0.95
t_value = t.ppf((1 + confidence) / 2., n - 2)
s_err = np.sqrt(np.sum((data_y - model.predict(data_x)) ** 2) / (n - 2))
y_err = t_value * s_err * np.sqrt(1 / n + (x_vals - np.mean(data_x)) ** 2 / np.sum((data_x - np.mean(data_x)) ** 2))

# Confidence interval shading
plt.fill_between(
    x_vals,
    y_pred - y_err,
    y_pred + y_err,
    color='MediumVioletRed',
    alpha=0.15
)

# Add scatter points
plt.scatter(
    data_x.flatten(),
    data_y,
    alpha=1,
    s=75,
    marker='^',  # Custom marker shape
    color='MediumVioletRed'
)

# Add statistics as a table
table_data = [
    ['Pearson Correlation', f'{pearson_corr:.2f}'],
    ['P-value', f'{p_value:.8f}'],
    ['R²', f'{r_squared:.2f}'],
    ['Adjusted R²', f'{adjusted_r_squared:.2f}'],
    ['95% CI', f'[{ci_lower:.2f}, {ci_upper:.2f}]']
]

plt.table(
    cellText=table_data,
    colLabels=['Statistic', 'Value'],
    cellLoc='center',
    loc='bottom',
    bbox=[0.0, -0.4, 1, 0.3]
)

plt.title('Scatter Plot with Regression and Statistics')
plt.xlabel('pNfH 6-BL')
plt.ylabel('Aß 1-42 6-BL')

plt.tight_layout()
plt.savefig('df_new_5_wo_labels.png', dpi=300)
plt.show()

In [ ]:
df_new_6 = pd.read_excel("correlation281124.xlsx", sheet_name=5, index_col=0)  # 0 refers to the first sheet
df_new_6.fillna(0, inplace=True)
df_new_6

In [ ]:
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import pearsonr, t
from sklearn.linear_model import LinearRegression
from adjustText import adjust_text

# Assuming df_new_6 is your DataFrame containing the data
data_x = df_new_6['NfL 6-BL'].values.reshape(-1, 1)
data_y = df_new_6['Aß 1-42 6-BL'].values

# Linear regression model
model = LinearRegression()
model.fit(data_x, data_y)
r_squared = model.score(data_x, data_y)
pearson_corr, p_value = pearsonr(data_x.flatten(), data_y)

# Calculate adjusted R-squared
n = len(data_x)  # Number of samples
p = 1  # Number of predictors
adjusted_r_squared = 1 - ((1 - r_squared) * (n - 1) / (n - p - 1))

# Bootstrap function for Pearson correlation confidence intervals
def bootstrap_pearsonr(data_x, data_y, n_bootstraps=1000, ci=95):
    bootstrapped_corrs = []
    for _ in range(n_bootstraps):
        sample_indices = np.random.choice(len(data_x), len(data_x), replace=True)
        sample_x = data_x[sample_indices]
        sample_y = data_y[sample_indices]
        boot_corr, _ = pearsonr(sample_x, sample_y)
        bootstrapped_corrs.append(boot_corr)
    lower_bound = np.percentile(bootstrapped_corrs, (100 - ci) / 2)
    upper_bound = np.percentile(bootstrapped_corrs, 100 - (100 - ci) / 2)
    return lower_bound, upper_bound

ci_lower, ci_upper = bootstrap_pearsonr(data_x.flatten(), data_y)

# Plot scatter and regression line
plt.figure(figsize=(8, 8))
sns.regplot(
    x=data_x.flatten(),
    y=data_y,
    scatter=False,
    line_kws={'color': 'black', 'linestyle': '--'},
    color='MediumVioletRed',
    ci=None
)

# Generate x-values for prediction and calculate confidence intervals
x_vals = np.linspace(data_x.min(), data_x.max(), 100)
y_pred = model.predict(x_vals.reshape(-1, 1))

confidence = 0.95
t_value = t.ppf((1 + confidence) / 2., n - 2)
s_err = np.sqrt(np.sum((data_y - model.predict(data_x)) ** 2) / (n - 2))
y_err = t_value * s_err * np.sqrt(1 / n + (x_vals - np.mean(data_x)) ** 2 / np.sum((data_x - np.mean(data_x)) ** 2))

# Confidence interval shading
plt.fill_between(
    x_vals,
    y_pred - y_err,
    y_pred + y_err,
    color='MediumVioletRed',
    alpha=0.15
)

# Add scatter points
plt.scatter(
    data_x.flatten(),
    data_y,
    alpha=1,
    s=75,
    marker='^',  # Custom marker shape
    color='MediumVioletRed'
)

# Add text annotations for points
text_annotations = []
x_offset = 0  # Horizontal offset
y_offset = 25   # Vertical offset

for index, _ in df_new_6.iterrows():
    text = plt.text(
        df_new_6.loc[index, 'NfL 6-BL'] + x_offset,
        df_new_6.loc[index, 'Aß 1-42 6-BL'] + y_offset,
        str(index),
        fontsize=10,
        fontweight='bold'
    )
    text_annotations.append(text)

# Automatically adjust text positions
adjust_text(
    text_annotations,
    arrowprops=dict(arrowstyle='->', color='black', lw=1.5)
)

# Add statistics as a table
table_data = [
    ['Pearson Correlation', f'{pearson_corr:.2f}'],
    ['P-value', f'{p_value:.8f}'],
    ['R²', f'{r_squared:.2f}'],
    ['Adjusted R²', f'{adjusted_r_squared:.2f}'],
    ['95% CI', f'[{ci_lower:.2f}, {ci_upper:.2f}]']
]

plt.table(
    cellText=table_data,
    colLabels=['Statistic', 'Value'],
    cellLoc='center',
    loc='bottom',
    bbox=[0.0, -0.4, 1, 0.3]
)

plt.title('Scatter Plot with Regression and Statistics')
plt.xlabel('NfL 6-BL')
plt.ylabel('Aß 1-42 6-BL')

plt.tight_layout()
plt.savefig('df_new_6.png', dpi=300)
plt.show()

In [ ]:
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import pearsonr, t
from sklearn.linear_model import LinearRegression
from adjustText import adjust_text

# Assuming df_new_6 is your DataFrame containing the data
data_x = df_new_6['NfL 6-BL'].values.reshape(-1, 1)
data_y = df_new_6['Aß 1-42 6-BL'].values

# Linear regression model
model = LinearRegression()
model.fit(data_x, data_y)
r_squared = model.score(data_x, data_y)
pearson_corr, p_value = pearsonr(data_x.flatten(), data_y)

# Calculate adjusted R-squared
n = len(data_x)  # Number of samples
p = 1  # Number of predictors
adjusted_r_squared = 1 - ((1 - r_squared) * (n - 1) / (n - p - 1))

# Bootstrap function for Pearson correlation confidence intervals
def bootstrap_pearsonr(data_x, data_y, n_bootstraps=1000, ci=95):
    bootstrapped_corrs = []
    for _ in range(n_bootstraps):
        sample_indices = np.random.choice(len(data_x), len(data_x), replace=True)
        sample_x = data_x[sample_indices]
        sample_y = data_y[sample_indices]
        boot_corr, _ = pearsonr(sample_x, sample_y)
        bootstrapped_corrs.append(boot_corr)
    lower_bound = np.percentile(bootstrapped_corrs, (100 - ci) / 2)
    upper_bound = np.percentile(bootstrapped_corrs, 100 - (100 - ci) / 2)
    return lower_bound, upper_bound

ci_lower, ci_upper = bootstrap_pearsonr(data_x.flatten(), data_y)

# Plot scatter and regression line
plt.figure(figsize=(8, 8))
sns.regplot(
    x=data_x.flatten(),
    y=data_y,
    scatter=False,
    line_kws={'color': 'black', 'linestyle': '--'},
    color='MediumVioletRed',
    ci=None
)

# Generate x-values for prediction and calculate confidence intervals
x_vals = np.linspace(data_x.min(), data_x.max(), 100)
y_pred = model.predict(x_vals.reshape(-1, 1))

confidence = 0.95
t_value = t.ppf((1 + confidence) / 2., n - 2)
s_err = np.sqrt(np.sum((data_y - model.predict(data_x)) ** 2) / (n - 2))
y_err = t_value * s_err * np.sqrt(1 / n + (x_vals - np.mean(data_x)) ** 2 / np.sum((data_x - np.mean(data_x)) ** 2))

# Confidence interval shading
plt.fill_between(
    x_vals,
    y_pred - y_err,
    y_pred + y_err,
    color='MediumVioletRed',
    alpha=0.15
)

# Add scatter points
plt.scatter(
    data_x.flatten(),
    data_y,
    alpha=1,
    s=75,
    marker='^',  # Custom marker shape
    color='MediumVioletRed'
)

# Add statistics as a table
table_data = [
    ['Pearson Correlation', f'{pearson_corr:.2f}'],
    ['P-value', f'{p_value:.8f}'],
    ['R²', f'{r_squared:.2f}'],
    ['Adjusted R²', f'{adjusted_r_squared:.2f}'],
    ['95% CI', f'[{ci_lower:.2f}, {ci_upper:.2f}]']
]

plt.table(
    cellText=table_data,
    colLabels=['Statistic', 'Value'],
    cellLoc='center',
    loc='bottom',
    bbox=[0.0, -0.4, 1, 0.3]
)

plt.title('Scatter Plot with Regression and Statistics')
plt.xlabel('NfL 6-BL')
plt.ylabel('Aß 1-42 6-BL')

plt.tight_layout()
plt.savefig('df_new_6_wo_labels.png', dpi=300)
plt.show()

In [ ]:
df_new_7 = pd.read_excel("correlation281124.xlsx", sheet_name=6, index_col=0)  # 0 refers to the first sheet
df_new_7.fillna(0, inplace=True)
df_new_7

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import pearsonr

# Assuming df1 is your DataFrame containing the data

# Plot the scatter plot with regression line and rho value
plt.figure(figsize=(8, 6))
sns.regplot(x=df_new_7['Score-R 6-BL'], y=df_new_7['pNfH 6-BL'], scatter_kws={'alpha':0.5}, line_kws={'color': 'black'}, color='blue')

# Calculate Pearson correlation coefficient and p-value
pearson_corr, p_value = pearsonr(df_new_7['Score-R 6-BL'], df_new_7['pNfH 6-BL'])

# Calculate R-squared value
r_squared = pearson_corr**2

# Add Pearson correlation coefficient, p-value, and R-squared value as text
plt.text(df_new_7['Score-R 6-BL'].min(), df_new_7['pNfH 6-BL'].max(), 
         'Pearson Correlation: {:.2f}\nP-value: {:.8f}\nR²: {:.2f}'.format(pearson_corr, p_value, r_squared),
         verticalalignment='top', horizontalalignment='left')

# Add labels to each point based on the index
for index, _ in df_new_7.iterrows():
    plt.text(df_new_7.loc[index, 'Score-R 6-BL'], df_new_7.loc[index, 'pNfH 6-BL'], str(index), fontsize=8)

plt.title('Scatter Plot')
plt.xlabel('Score-R 6-BL')
plt.ylabel('pNfH 6-BL')

plt.savefig('df_new_7.png')

# Show plot
plt.show()

In [ ]:
df_new_8 = pd.read_excel("correlation281124.xlsx", sheet_name=7, index_col=0)  # 0 refers to the first sheet
df_new_8.fillna(0, inplace=True)
df_new_8

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import pearsonr

# Assuming df1 is your DataFrame containing the data

# Plot the scatter plot with regression line and rho value
plt.figure(figsize=(8, 6))
sns.regplot(x=df_new_8['Score-R 6-BL'], y=df_new_8['NfL 6-BL'], scatter_kws={'alpha':0.5}, line_kws={'color': 'black'}, color='blue')

# Calculate Pearson correlation coefficient and p-value
pearson_corr, p_value = pearsonr(df_new_8['Score-R 6-BL'], df_new_8['NfL 6-BL'])

# Calculate R-squared value
r_squared = pearson_corr**2

# Add Pearson correlation coefficient, p-value, and R-squared value as text
plt.text(df_new_8['Score-R 6-BL'].min(), df_new_8['NfL 6-BL'].max(), 
         'Pearson Correlation: {:.2f}\nP-value: {:.8f}\nR²: {:.2f}'.format(pearson_corr, p_value, r_squared),
         verticalalignment='top', horizontalalignment='left')

# Add labels to each point based on the index
for index, _ in df_new_8.iterrows():
    plt.text(df_new_8.loc[index, 'Score-R 6-BL'], df_new_8.loc[index, 'NfL 6-BL'], str(index), fontsize=8)

plt.title('Scatter Plot')
plt.xlabel('Score-R 6-BL')
plt.ylabel('NfL 6-BL')

plt.savefig('df_new_8.png')

# Show plot
plt.show()

In [ ]:
df_new_9 = pd.read_excel("correlation281124.xlsx", sheet_name=8, index_col=0)  # 0 refers to the first sheet
df_new_9.fillna(0, inplace=True)
df_new_9

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import pearsonr

# Assuming df1 is your DataFrame containing the data

# Plot the scatter plot with regression line and rho value
plt.figure(figsize=(8, 6))
sns.regplot(x=df_new_9['Score-R 6-BL'], y=df_new_9['UCHL1 6-BL'], scatter_kws={'alpha':0.5}, line_kws={'color': 'black'}, color='blue')

# Calculate Pearson correlation coefficient and p-value
pearson_corr, p_value = pearsonr(df_new_9['Score-R 6-BL'], df_new_9['UCHL1 6-BL'])

# Calculate R-squared value
r_squared = pearson_corr**2

# Add Pearson correlation coefficient, p-value, and R-squared value as text
plt.text(df_new_9['Score-R 6-BL'].min(), df_new_9['UCHL1 6-BL'].max(), 
         'Pearson Correlation: {:.2f}\nP-value: {:.8f}\nR²: {:.2f}'.format(pearson_corr, p_value, r_squared),
         verticalalignment='top', horizontalalignment='left')

# Add labels to each point based on the index
for index, _ in df_new_9.iterrows():
    plt.text(df_new_9.loc[index, 'Score-R 6-BL'], df_new_9.loc[index, 'UCHL1 6-BL'], str(index), fontsize=8)

plt.title('Scatter Plot')
plt.xlabel('Score-R 6-BL')
plt.ylabel('UCHL1 6-BL')

plt.savefig('df_new_9.png')

# Show plot
plt.show()

In [ ]:
df_new_10 = pd.read_excel("correlation281124.xlsx", sheet_name=9, index_col=0)  # 0 refers to the first sheet
df_new_10.fillna(0, inplace=True)
df_new_10

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import pearsonr

# Assuming df1 is your DataFrame containing the data

# Plot the scatter plot with regression line and rho value
plt.figure(figsize=(8, 6))
sns.regplot(x=df_new_10['Score-R 6-BL'], y=df_new_10['Aß 1-40 6-BL'], scatter_kws={'alpha':0.5}, line_kws={'color': 'black'}, color='blue')

# Calculate Pearson correlation coefficient and p-value
pearson_corr, p_value = pearsonr(df_new_10['Score-R 6-BL'], df_new_10['Aß 1-40 6-BL'])

# Calculate R-squared value
r_squared = pearson_corr**2

# Add Pearson correlation coefficient, p-value, and R-squared value as text
plt.text(df_new_10['Score-R 6-BL'].min(), df_new_10['Aß 1-40 6-BL'].max(), 
         'Pearson Correlation: {:.2f}\nP-value: {:.8f}\nR²: {:.2f}'.format(pearson_corr, p_value, r_squared),
         verticalalignment='top', horizontalalignment='left')

# Add labels to each point based on the index
for index, _ in df_new_10.iterrows():
    plt.text(df_new_10.loc[index, 'Score-R 6-BL'], df_new_10.loc[index, 'Aß 1-40 6-BL'], str(index), fontsize=8)

plt.title('Scatter Plot')
plt.xlabel('Score-R 6-BL')
plt.ylabel('Aß 1-40 6-BL')

plt.savefig('df_new_10.png')

# Show plot
plt.show()

In [ ]:
df_new_11 = pd.read_excel("correlation281124.xlsx", sheet_name=10, index_col=0)  # 0 refers to the first sheet
df_new_11.fillna(0, inplace=True)
df_new_11

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import pearsonr

# Assuming df1 is your DataFrame containing the data

# Plot the scatter plot with regression line and rho value
plt.figure(figsize=(8, 6))
sns.regplot(x=df_new_11['Score-R 6-BL'], y=df_new_11['Aß 1-42 6-BL'], scatter_kws={'alpha':0.5}, line_kws={'color': 'black'}, color='blue')

# Calculate Pearson correlation coefficient and p-value
pearson_corr, p_value = pearsonr(df_new_11['Score-R 6-BL'], df_new_11['Aß 1-42 6-BL'])

# Calculate R-squared value
r_squared = pearson_corr**2

# Add Pearson correlation coefficient, p-value, and R-squared value as text
plt.text(df_new_11['Score-R 6-BL'].min(), df_new_11['Aß 1-42 6-BL'].max(), 
         'Pearson Correlation: {:.2f}\nP-value: {:.8f}\nR²: {:.2f}'.format(pearson_corr, p_value, r_squared),
         verticalalignment='top', horizontalalignment='left')

# Add labels to each point based on the index
for index, _ in df_new_11.iterrows():
    plt.text(df_new_11.loc[index, 'Score-R 6-BL'], df_new_11.loc[index, 'Aß 1-42 6-BL'], str(index), fontsize=8)

plt.title('Scatter Plot')
plt.xlabel('Score-R 6-BL')
plt.ylabel('Aß 1-42 6-BL')

plt.savefig('df_new_11.png')

# Show plot
plt.show()

In [ ]:
df_new_12 = pd.read_excel("correlation031224 male.xlsx", sheet_name=0, index_col=0)  # 0 refers to the first sheet
df_new_12.fillna(0, inplace=True)
df_new_12

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import pearsonr

# Assuming df1 is your DataFrame containing the data

# Plot the scatter plot with regression line and rho value
plt.figure(figsize=(8, 6))
sns.regplot(x=df_new_12['pNfH 6-BL'], y=df_new_12['Aß 1-40 6-BL'], scatter_kws={'alpha':0.5}, line_kws={'color': 'black'}, color='blue')

# Calculate Pearson correlation coefficient and p-value
pearson_corr, p_value = pearsonr(df_new_12['pNfH 6-BL'], df_new_12['Aß 1-40 6-BL'])

# Calculate R-squared value
r_squared = pearson_corr**2

# Add Pearson correlation coefficient, p-value, and R-squared value as text
plt.text(df_new_12['pNfH 6-BL'].min(), df_new_12['Aß 1-40 6-BL'].max(), 
         'Pearson Correlation: {:.2f}\nP-value: {:.8f}\nR²: {:.2f}'.format(pearson_corr, p_value, r_squared),
         verticalalignment='top', horizontalalignment='left')

# Add labels to each point based on the index
for index, _ in df_new_12.iterrows():
    plt.text(df_new_12.loc[index, 'pNfH 6-BL'], df_new_12.loc[index, 'Aß 1-40 6-BL'], str(index), fontsize=8)

plt.title('Scatter Plot')
plt.xlabel('pNfH 6-BL')
plt.ylabel('Aß 1-40 6-BL')

plt.savefig('df_new_12.png')

# Show plot
plt.show()

In [ ]:
df_new_13 = pd.read_excel("correlation031224 male.xlsx", sheet_name=1, index_col=0)  # 0 refers to the first sheet
df_new_13.fillna(0, inplace=True)
df_new_13

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import pearsonr

# Assuming df1 is your DataFrame containing the data

# Plot the scatter plot with regression line and rho value
plt.figure(figsize=(8, 6))
sns.regplot(x=df_new_13['NfL 6-BL'], y=df_new_13['Aß 1-40 6-BL'], scatter_kws={'alpha':0.5}, line_kws={'color': 'black'}, color='blue')

# Calculate Pearson correlation coefficient and p-value
pearson_corr, p_value = pearsonr(df_new_13['NfL 6-BL'], df_new_13['Aß 1-40 6-BL'])

# Calculate R-squared value
r_squared = pearson_corr**2

# Add Pearson correlation coefficient, p-value, and R-squared value as text
plt.text(df_new_13['NfL 6-BL'].min(), df_new_13['Aß 1-40 6-BL'].max(), 
         'Pearson Correlation: {:.2f}\nP-value: {:.8f}\nR²: {:.2f}'.format(pearson_corr, p_value, r_squared),
         verticalalignment='top', horizontalalignment='left')

# Add labels to each point based on the index
for index, _ in df_new_13.iterrows():
    plt.text(df_new_13.loc[index, 'NfL 6-BL'], df_new_13.loc[index, 'Aß 1-40 6-BL'], str(index), fontsize=8)

plt.title('Scatter Plot')
plt.xlabel('NfL 6-BL')
plt.ylabel('Aß 1-40 6-BL')

plt.savefig('df_new_13.png')

# Show plot
plt.show()

In [ ]:
df_new_14 = pd.read_excel("correlation031224 male.xlsx", sheet_name=2, index_col=0)  # 0 refers to the first sheet
df_new_14.fillna(0, inplace=True)
df_new_14

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import pearsonr

# Assuming df1 is your DataFrame containing the data

# Plot the scatter plot with regression line and rho value
plt.figure(figsize=(8, 6))
sns.regplot(x=df_new_14['pNfH 6-BL'], y=df_new_14['Aß 1-42 6-BL'], scatter_kws={'alpha':0.5}, line_kws={'color': 'black'}, color='blue')

# Calculate Pearson correlation coefficient and p-value
pearson_corr, p_value = pearsonr(df_new_14['pNfH 6-BL'], df_new_14['Aß 1-42 6-BL'])

# Calculate R-squared value
r_squared = pearson_corr**2

# Add Pearson correlation coefficient, p-value, and R-squared value as text
plt.text(df_new_14['pNfH 6-BL'].min(), df_new_14['Aß 1-42 6-BL'].max(), 
         'Pearson Correlation: {:.2f}\nP-value: {:.8f}\nR²: {:.2f}'.format(pearson_corr, p_value, r_squared),
         verticalalignment='top', horizontalalignment='left')

# Add labels to each point based on the index
for index, _ in df_new_14.iterrows():
    plt.text(df_new_14.loc[index, 'pNfH 6-BL'], df_new_14.loc[index, 'Aß 1-42 6-BL'], str(index), fontsize=8)

plt.title('Scatter Plot')
plt.xlabel('pNfH 6-BL')
plt.ylabel('Aß 1-42 6-BL')

plt.savefig('df_new_14.png')

# Show plot
plt.show()

In [ ]:
df_new_15 = pd.read_excel("correlation031224 male.xlsx", sheet_name=3, index_col=0)  # 0 refers to the first sheet
df_new_15.fillna(0, inplace=True)
df_new_15

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import pearsonr

# Assuming df1 is your DataFrame containing the data

# Plot the scatter plot with regression line and rho value
plt.figure(figsize=(8, 6))
sns.regplot(x=df_new_15['NfL 6-BL'], y=df_new_15['Aß 1-42 6-BL'], scatter_kws={'alpha':0.5}, line_kws={'color': 'black'}, color='blue')

# Calculate Pearson correlation coefficient and p-value
pearson_corr, p_value = pearsonr(df_new_15['NfL 6-BL'], df_new_15['Aß 1-42 6-BL'])

# Calculate R-squared value
r_squared = pearson_corr**2

# Add Pearson correlation coefficient, p-value, and R-squared value as text
plt.text(df_new_15['NfL 6-BL'].min(), df_new_15['Aß 1-42 6-BL'].max(), 
         'Pearson Correlation: {:.2f}\nP-value: {:.8f}\nR²: {:.2f}'.format(pearson_corr, p_value, r_squared),
         verticalalignment='top', horizontalalignment='left')

# Add labels to each point based on the index
for index, _ in df_new_15.iterrows():
    plt.text(df_new_15.loc[index, 'NfL 6-BL'], df_new_15.loc[index, 'Aß 1-42 6-BL'], str(index), fontsize=8)

plt.title('Scatter Plot')
plt.xlabel('NfL 6-BL')
plt.ylabel('Aß 1-42 6-BL')

plt.savefig('df_new_15.png')

# Show plot
plt.show()